# 📊 SÍNTESE VISUAL 4 — Resumo Final: Evolução + Ablation + Comparativo

In [ ]:
import json, sys, numpy as np
sys.path.insert(0, '..')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

METRICS_DIR = Path('experiments_results/metrics')
FIGURES_DIR = Path('experiments_results/figures')

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#f0f6fc','axes.labelcolor':'#f0f6fc',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'axes.edgecolor':'#30363d','grid.color':'#30363d','grid.alpha':0.5,
})

def load(nb_id):
    p = METRICS_DIR/f'{nb_id}_results.json'
    return json.load(open(p)) if p.exists() else {}

def best_pauc(data):
    for key in ['avg_prot_a','pauc01']:
        if key in data: return float(data[key])
    if 'protocol_a' in data: return float(data['protocol_a'].get('pauc01',0))
    if 'spec' in data: return float(data['spec'].get('avg_prot_a',0))
    if 'models' in data:
        vals = [v.get('pauc01',0) for v in data['models'].values() if isinstance(v,dict)]
        if vals: return float(max(vals))
    if 'model' in data:
        m = data['model']
        if isinstance(m,dict): return float(m.get('pauc01',0))
    if 'ranking' in data:
        vals = [v.get('pauc01',0) for v in data['ranking'].values()]
        if vals: return float(max(vals))
    return 0.0

all_data = {f'nb{i:02d}': load(f'nb{i:02d}') for i in range(1,29)}
print(f"✅ {sum(1 for v in all_data.values() if v)} resultados carregados")


In [ ]:
# Evolução pAUC@0.1 ao longo dos NB
nbs = [f'nb{i:02d}' for i in range(1,29)]
pauc_vals = [best_pauc(all_data[nb]) for nb in nbs]
labels = [nb.upper() for nb in nbs]

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0d1117')

# 1) Linha de evolução
ax1 = fig.add_subplot(2, 2, (1,2))
ax1.set_facecolor('#161b22')
x = np.arange(len(labels))
ax1.plot(x, pauc_vals, color='#2ea043', marker='o', lw=2, ms=5, zorder=3)
ax1.fill_between(x, pauc_vals, 0.80,
                  where=np.array(pauc_vals)>=0.80, alpha=0.1, color='#2ea043')
ax1.axhline(0.80,color='#f0883e',ls='--',lw=1.5,alpha=0.8,label='Limite 0.80')
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=60, ha='right', fontsize=7)
ax1.set_ylim(0,1.05); ax1.set_ylabel('pAUC@0.1')
ax1.set_title('Evolução pAUC@0.1: NB01 → NB28', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True,alpha=0.3)

# 2) Ablation
nb27 = load('nb27')
ablation = nb27.get('ablation', {})
pauc_full = nb27.get('pauc_full', 0)
if ablation:
    comps = [k for k in ablation if k != 'Referência (completo)']
    deltas = [ablation[k].get('delta',0) for k in comps]
    colors_ab = ['#da3633' if d>=0.05 else '#f0883e' if d>=0.02 else '#2ea043' for d in deltas]
    ax2 = fig.add_subplot(2, 2, 3)
    ax2.set_facecolor('#161b22')
    short_comps = [c.replace('Sem ','') for c in comps]
    ax2.barh(short_comps, deltas, color=colors_ab, alpha=0.85)
    ax2.set_xlabel('Δ pAUC@0.1')
    ax2.set_title(f'Ablation Study (ref={pauc_full:.3f})', fontsize=11, fontweight='bold')
    ax2.grid(True,axis='x',alpha=0.3)

# 3) Final comparison
nb28 = load('nb28')
spec = nb28.get('spec', {})
ax3 = fig.add_subplot(2, 2, 4)
ax3.set_facecolor('#161b22')
final_models = ['Prot.A\n(XGBoost)', 'Prot.B\n(GMM)', 'DCASE\nScore']
final_vals = [spec.get('avg_prot_a',0), spec.get('avg_prot_b',0), spec.get('avg_dcase_score',0)]
bars3 = ax3.bar(final_models, final_vals, color=['#2ea043','#1f6feb','#f0883e'], alpha=0.9)
ax3.axhline(0.80,color='white',ls='--',lw=1.2,alpha=0.6,label='Limite 0.80')
for bar, val in zip(bars3, final_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, val+0.01, f'{val:.3f}',
             ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
ax3.set_ylim(0,1.1); ax3.set_title('DIAMOND CRISTAL CLEAN\nResultado Final (LOSO)', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True,axis='y',alpha=0.3)

plt.tight_layout(pad=2)
plt.savefig('experiments_results/figures/visual4_resumo_final.png', dpi=130,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("✅ Resumo visual completo gerado")
